In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform
from scipy.stats import norm
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from statsmodels.tsa.ar_model import AutoReg
import geopandas as gpd
import warnings

/opt/anaconda3/envs/env_master_2026/lib/python3.13/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


In [2]:
df=pd.read_parquet('/.venv/Data/merged_adm1_wide_normalized.parquet')

In [3]:
df.columns

Index(['Country', 'location_name_full', 'Level 1', 'Area', 'adm1_pcode',
       'adm2_pcode', 'From', 'To', 'Validity period', 'Date of analysis',
       'admin_level', 'resource_hdx_id', 'phase_1_number',
       'phase_1_percentage', 'phase_2_number', 'phase_2_percentage',
       'phase_3_number', 'phase_3_percentage', 'phase_3plus_number',
       'phase_3plus_percentage', 'phase_4_number', 'phase_4_percentage',
       'phase_5_number', 'phase_5_percentage', 'phase_all_number',
       'phase_all_percentage', 'reference_period_end',
       'acled_civilian_targeting_events', 'acled_demonstration_events',
       'acled_political_violence_events',
       'acled_civilian_targeting_fatalities', 'acled_demonstration_fatalities',
       'acled_political_violence_fatalities', 'acled_total_events',
       'acled_total_fatalities', 'idp_population', 'idp_assessment_type',
       'idp_reporting_round', 'idp_staleness_days', 'rain_1m_sum', 'rain_1m',
       'rain_3m', 'rain_anomaly_1m', 'rain_anom

In [4]:
df_geo=pd.read_parquet('/.venv/Data/merged_adm1_wide_con_coordinate.parquet')

In [5]:
df_geo.columns

Index(['Country', 'location_name_full', 'Level 1', 'Area', 'adm1_pcode',
       'adm2_pcode', 'From', 'To', 'Validity period', 'Date of analysis',
       'admin_level', 'resource_hdx_id', 'phase_1_number',
       'phase_1_percentage', 'phase_2_number', 'phase_2_percentage',
       'phase_3_number', 'phase_3_percentage', 'phase_3plus_number',
       'phase_3plus_percentage', 'phase_4_number', 'phase_4_percentage',
       'phase_5_number', 'phase_5_percentage', 'phase_all_number',
       'phase_all_percentage', 'reference_period_end',
       'acled_civilian_targeting_events', 'acled_demonstration_events',
       'acled_political_violence_events',
       'acled_civilian_targeting_fatalities', 'acled_demonstration_fatalities',
       'acled_political_violence_fatalities', 'acled_total_events',
       'acled_total_fatalities', 'idp_population', 'idp_assessment_type',
       'idp_reporting_round', 'idp_staleness_days', 'rain_1m_sum', 'rain_1m',
       'rain_3m', 'rain_anomaly_1m', 'rain_anom

In [6]:
# 1. Selezioniamo solo il codice PCODE 1 e le coordinate da df_geo
# Rimuoviamo i duplicati per avere una corrispondenza 1-a-1 per ogni codice
df_geo_clean = df_geo[['adm1_pcode', 'latitude', 'longitude']].drop_duplicates(subset=['adm1_pcode'])

# 2. Facciamo il merge su df usando 'adm1_pcode' come chiave
df = df.merge(df_geo_clean, on='adm1_pcode', how='left')

In [7]:
from Funzioni_clustering import compute_similarity_matrix
from Funzioni_clustering import run_clustering_flow
from Funzioni_clustering import z_score_normalize
from Funzioni_clustering import calculate_approx_entropy
from Funzioni_clustering import calculate_hurst
from Funzioni_clustering import extract_structural_features
from Funzioni_clustering import dtw_distance

In [8]:
# Group by Country and adm1_pcode to resolve name and coordinate duplicates per region
regions_data = {}
metadata_rows = []

grouped = df.groupby(["Country", "adm1_pcode"])
for (country, pcode), group in grouped:
    # Resolve region name (first non-null Level 1)
    names = group["Level 1"].dropna()
    r_name = names.iloc[0] if not names.empty else "Unknown Region"

    # Resolve coordinates (mean of non-null)
    lats = group["latitude"].dropna()
    lons = group["longitude"].dropna()
    lat = lats.mean() if not lats.empty else np.nan
    lon = lons.mean() if not lons.empty else np.nan

    # Extract and expand time series using all validity periods
    expanded = []
    for _, row in group.iterrows():
        m_range = pd.date_range(start=row["From"], end=row["To"], freq="MS")
        for m in m_range:
            expanded.append({"date": m, "pct": row["phase_3plus_percentage"]})

    if expanded:
        # Group by date and take mean to aggregate overlapping assessments (current/projections)
        df_ipc = pd.DataFrame(expanded).groupby("date")["pct"].mean().to_frame()
        # Reindex to regular monthly grid from min to max date
        full_range = pd.date_range(start=df_ipc.index.min(), end=df_ipc.index.max(), freq="MS")
        df_ipc = df_ipc.reindex(full_range)
        # Interpolate and ffill/bfill gaps
        df_ipc = df_ipc.interpolate(method="linear").ffill().bfill()

        # Check if the series meets the minimum length constraint (>= 24 months) and has no NaNs
        if len(df_ipc) >= 24 and not df_ipc["pct"].isna().any():
            regions_data[f"{country}_{pcode}"] = df_ipc["pct"]
            metadata_rows.append({
                "country": country,
                "adm1_pcode": pcode,
                "region_name": r_name,
                "latitude": lat,
                "longitude": lon,
                "key": f"{country}_{pcode}",
                "series_length": len(df_ipc)
            })

df_meta = pd.DataFrame(metadata_rows)

In [9]:
print("\n--- Step 2: Feature Extraction ---")
features_list = []
for idx, row in df_meta.iterrows():
    key = row["key"]
    series = regions_data[key]
    feats = extract_structural_features(series)
    feats.update({
        "country": row["country"],
        "adm1_pcode": row["adm1_pcode"],
        "region_name": row["region_name"],
        "latitude": row["latitude"],
        "longitude": row["longitude"]
    })
    features_list.append(feats)

df_features = pd.DataFrame(features_list)
feature_cols = ["stat_mean", "stat_var", "stat_skew", "stat_kurt", "hurst_exponent", "approx_entropy", "ar1_coeff", "ar2_coeff", "ar3_coeff"]

print("Extracted structural features example:")
print(df_features[["country", "adm1_pcode", "region_name"] + feature_cols[:3]].head())


--- Step 2: Feature Extraction ---
Extracted structural features example:
  country adm1_pcode    region_name  stat_mean    stat_var  stat_skew
0     AFG       AF01          Kabul  29.010101  116.699065   0.513777
1     AFG       AF02         Kapisa  25.590909   73.826307   0.841591
2     AFG       AF03         Parwan  26.338384  119.304868   0.751768
3     AFG       AF04  Maidan Wardak  34.924242  101.664176   0.075339
4     AFG       AF05          Logar  27.505051   94.314884   0.509401


In [10]:
import os
import warnings

# Silenzia i warning superflui per mantenere i log puliti
warnings.filterwarnings("ignore")

# 1. Trova la cartella corrente (dove si trova questo file/notebook)
cwd = os.path.dirname(os.path.abspath(__file__)) if "__file__" in locals() else os.getcwd()

# 2. Si sposta verso l'alto nell'albero delle cartelle di progetto:
# ml_dir = Sale di 1 livello rispetto a CWD (cartella principale del Machine Learning)
ml_dir = os.path.abspath(os.path.join(cwd, ".."))

# hero_dir = Sale di un ulteriore livello (la cartella radice del progetto globale "HERO")
hero_dir = os.path.abspath(os.path.join(ml_dir, ".."))

# 3. Definisce i percorsi specifici per i Dati e per i Risultati
data_dir = os.path.join(hero_dir, "data")
merged_parquet = os.path.join(data_dir, "merged", "merged_adm1_wide_con_coordinate.parquet")
results_dir = os.path.join(ml_dir, "results")

# 4. Crea fisicamente le cartelle sul PC se non esistono ancora
os.makedirs(results_dir, exist_ok=True)
os.makedirs(os.path.join(results_dir, "global"), exist_ok=True)

# 5. Stampa di controllo per verificare che la struttura sia corretta
print(f"ML Directory: {ml_dir}")
print(f"Results Directory: {results_dir}")
print(f"Data File: {merged_parquet}")

ML Directory: /Users/mattiadetommaso/PyCharmMiscProject/.venv
Results Directory: /Users/mattiadetommaso/PyCharmMiscProject/.venv/results
Data File: /Users/mattiadetommaso/PyCharmMiscProject/data/merged/merged_adm1_wide_con_coordinate.parquet


In [11]:
print("\n--- Step 4: Running Country-Level Analysis ---")
metrics_records = []

# Definiamo la lista delle feature nel notebook
feature_cols = ["stat_mean", "stat_var", "stat_skew", "stat_kurt", "hurst_exponent", "approx_entropy", "ar1_coeff", "ar2_coeff", "ar3_coeff"]

# Trova i paesi con almeno 4 regioni potenziali
country_counts = df_meta["country"].value_counts()
eligible_countries = country_counts[country_counts >= 4].index.tolist()
print(f"Eligible countries (>=4 regions): {eligible_countries}")

for c_code in eligible_countries:
    print(f"\nProcessing Country: {c_code}")

    try:
        # 1. Estrai le feature del paese corrente
        c_df_feat = df_features[df_features["country"] == c_code].reset_index(drop=True)
        n_samples = len(c_df_feat)

        # Controllo di sicurezza sulle regioni effettivamente presenti nel DataFrame delle feature
        if n_samples < 4:
            print(f"Skipping {c_code}: solo {n_samples} regioni valide in df_features (richieste almeno 4).")
            continue

        # 2. FIX: Costruisci c_series_dict basandoti SOLO sulle regioni realmente presenti in c_df_feat
        # Questo evita disallineamenti dimensionali tra le feature strutturali e le serie grezze per il DTW
        c_series_dict = {}
        for _, row in c_df_feat.iterrows():
            pcode = row["adm1_pcode"]
            # Recupera la chiave originaria da df_meta per estrarre la serie storica corretta
            meta_match = df_meta[df_meta["adm1_pcode"] == pcode]
            if not meta_match.empty:
                key = meta_match.iloc[0]["key"]
                if key in regions_data:
                    c_series_dict[pcode] = regions_data[key]

        # Verifica che il dizionario delle serie sia allineato
        if len(c_series_dict) != n_samples:
            print(f"Warning per {c_code}: Trovate {len(c_series_dict)} serie storiche per {n_samples} record di feature.")

        # Definizione cartelle di output
        c_results_dir = os.path.join(results_dir, c_code)
        os.makedirs(c_results_dir, exist_ok=True)
        out_prefix = os.path.join(c_results_dir, f"{c_code}_clustering")

        # 3. FIX: Calcolo protetto e stabile del numero ottimale di cluster (K)
        if n_samples <= 5:
            n_cl = 2
        else:
            n_cl = min(4, n_samples - 2)

        print(f"Running clustering with K={n_cl} for {n_samples} regions...")

        # Esecuzione del flusso di clustering passando esplicitamente feature_cols
        flow_results = run_clustering_flow(
            c_df_feat,
            c_series_dict,
            out_prefix,
            n_clusters=n_cl,
            is_global=False,
            feature_cols=feature_cols
        )

        # Salva il DataFrame con le etichette dei cluster in formato CSV
        flow_results["labels_df"].to_csv(out_prefix + "_labels.csv", index=False)

        # Estrazione e salvataggio delle metriche di performance (Silhouette score)
        mets = flow_results["metrics"]
        mets["country"] = c_code
        mets["num_regions"] = n_samples
        metrics_records.append(mets)
        print(f"Country {c_code} completed. Silhouette K-Means (No Coords): {mets['sil_km_no_coords']:.3f} | (With Coords): {mets['sil_km_with_coords']:.3f}")

        # --- Generazione delle Mappe Choropleth ---
        geojson_path = os.path.join(data_dir, "boundaries", c_code.lower(), f"{c_code.lower()}_admin1.geojson")
        if os.path.exists(geojson_path):
            try:
                gdf = gpd.read_file(geojson_path)

                # Identifica la colonna del codice PCODE (case-insensitive)
                pcode_col = None
                for col in gdf.columns:
                    if col.lower() == 'adm1_pcode':
                        pcode_col = col
                        break

                if pcode_col is not None:
                    gdf_merged = gdf.merge(flow_results["labels_df"], left_on=pcode_col, right_on="adm1_pcode", how="left")

                    # Plot affiancato (K-Means con e senza Coordinate geografiche)
                    fig, axes = plt.subplots(1, 2, figsize=(18, 9))

                    gdf_merged.plot(column="kmeans_features_no_coords", cmap="tab10", legend=True, categorical=True,
                                    ax=axes[0], missing_kwds={"color": "lightgrey"})
                    axes[0].set_title(f"{c_code} - K-Means (NO Coordinates)")
                    axes[0].axis("off")

                    gdf_merged.plot(column="kmeans_features_with_coords", cmap="tab10", legend=True, categorical=True,
                                    ax=axes[1], missing_kwds={"color": "lightgrey"})
                    axes[1].set_title(f"{c_code} - K-Means (WITH Coordinates)")
                    axes[1].axis("off")

                    plt.tight_layout()
                    plt.savefig(out_prefix + "_choropleth_comparison.png", dpi=150)
                    plt.close()
                    print(f"Saved choropleth map comparison for {c_code}")
                else:
                    print(f"Warning: Column 'adm1_pcode' not found in GeoJSON columns for {c_code}.")
            except Exception as geo_err:
                print(f"Choropleth mapping failed for {c_code}: {geo_err}")

    except Exception as country_error:
        print(f"ERROR: Errore critico durante l'elaborazione di {c_code}: {country_error}")
        continue


--- Step 4: Running Country-Level Analysis ---
Eligible countries (>=4 regions): ['AFG', 'CIV', 'NGA', 'COD', 'KEN', 'CPV', 'GTM', 'SOM', 'SDN', 'HND', 'CAF', 'GHA', 'SEN', 'NAM', 'LBR', 'BEN', 'MRT', 'MOZ', 'HTI', 'CMR', 'SSD', 'GNB', 'MLI', 'ZWE', 'NER', 'GMB', 'SLV', 'BGD', 'GIN', 'ETH', 'TGO', 'SLE', 'SWZ']

Processing Country: AFG
Running clustering with K=4 for 34 regions...
Country AFG completed. Silhouette K-Means (No Coords): 0.215 | (With Coords): 0.168

Processing Country: CIV
Running clustering with K=4 for 31 regions...
Country CIV completed. Silhouette K-Means (No Coords): 0.264 | (With Coords): 0.219

Processing Country: NGA
Running clustering with K=4 for 27 regions...
Country NGA completed. Silhouette K-Means (No Coords): 0.181 | (With Coords): 0.127

Processing Country: COD
Running clustering with K=4 for 26 regions...
Country COD completed. Silhouette K-Means (No Coords): 0.202 | (With Coords): 0.205

Processing Country: KEN
Running clustering with K=4 for 23 region

In [12]:
print("\n--- Step 5: Running Global Region-Level Clustering ---")
global_out_prefix = os.path.join(results_dir, "global", "global_regions")

# Run global clustering on features (K=5)
global_results = run_clustering_flow(df_features, None, global_out_prefix, n_clusters=5, is_global=True)
global_results["labels_df"].to_csv(global_out_prefix + "_labels.csv", index=False)

mets_global = global_results["metrics"]
print(f"Global Region Clustering completed.")
print(f"Silhouette Hierarchical (No Coords): {mets_global['sil_hier_no_coords']:.3f} | (With Coords): {mets_global['sil_hier_with_coords']:.3f}")
print(f"Silhouette K-Means      (No Coords): {mets_global['sil_km_no_coords']:.3f} | (With Coords): {mets_global['sil_km_with_coords']:.3f}")

# Plot Global Map
print("Attempting to plot Global Map...")
try:
    world_url = 'https://raw.githubusercontent.com/datasets/geo-boundaries-world-110m/master/countries.geojson'
    world = gpd.read_file(world_url)
    world = world[world["continent"] != "Antarctica"]

    # Load all boundaries for countries with valid regions
    gdfs = []
    for c in df_features["country"].unique():
        path = os.path.join(data_dir, "boundaries", c.lower(), f"{c.lower()}_admin1.geojson")
        if os.path.exists(path):
            try:
                gdf_c = gpd.read_file(path)
                pcode_col = None
                for col in gdf_c.columns:
                    if col.lower() == 'adm1_pcode':
                        pcode_col = col
                        break
                if pcode_col is not None:
                    gdf_c = gdf_c[[pcode_col, "geometry"]]
                    gdf_c = gdf_c.rename(columns={pcode_col: "adm1_pcode"})
                    gdfs.append(gdf_c)
            except Exception as e:
                pass

    if gdfs:
        all_regions_gdf = pd.concat(gdfs, ignore_index=True)
        all_regions_gdf = all_regions_gdf.merge(global_results["labels_df"], on="adm1_pcode", how="left")

        # 1. Global map for No Coords
        fig, ax = plt.subplots(figsize=(18, 10))
        world.plot(ax=ax, color="#f2f2f2", edgecolor="#d9d9d9")
        all_regions_gdf.dropna(subset=["kmeans_features_no_coords"]).plot(
            column="kmeans_features_no_coords", cmap="tab10", legend=True, categorical=True,
            ax=ax, legend_kwds={"bbox_to_anchor": (1.05, 1), "loc": "upper left"}
        )
        ax.set_title("Global Admin1 Region Clusters (PCA + K-Means - NO Coordinates)", fontsize=16)
        ax.set_xlim([-120, 150])
        ax.set_ylim([-40, 60])
        plt.tight_layout()
        plt.savefig(global_out_prefix + "_map_no_coords.png", dpi=150)
        plt.close()

        # 2. Global map for With Coords
        fig, ax = plt.subplots(figsize=(18, 10))
        world.plot(ax=ax, color="#f2f2f2", edgecolor="#d9d9d9")
        all_regions_gdf.dropna(subset=["kmeans_features_with_coords"]).plot(
            column="kmeans_features_with_coords", cmap="tab10", legend=True, categorical=True,
            ax=ax, legend_kwds={"bbox_to_anchor": (1.05, 1), "loc": "upper left"}
        )
        ax.set_title("Global Admin1 Region Clusters (PCA + K-Means - WITH Coordinates)", fontsize=16)
        ax.set_xlim([-120, 150])
        ax.set_ylim([-40, 60])
        plt.tight_layout()
        plt.savefig(global_out_prefix + "_map_with_coords.png", dpi=150)
        plt.close()
        print("Global maps plotted successfully.")
except Exception as e:
    print(f"Global region choropleth map plotting failed: {e}")


--- Step 5: Running Global Region-Level Clustering ---
Global Region Clustering completed.
Silhouette Hierarchical (No Coords): 0.145 | (With Coords): 0.128
Silhouette K-Means      (No Coords): 0.194 | (With Coords): 0.174
Attempting to plot Global Map...


ERROR 1: PROJ: proj_create_from_database: /opt/anaconda3/share/proj/proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 2 whereas a number >= 6 is expected. It comes from another PROJ installation.


Global region choropleth map plotting failed: Invalid projection: EPSG:4326: (Internal Proj Error: proj_create: no database context specified)


In [15]:
print("\n--- Step 6: Running Global National-Level Clustering ---")

# --- CARICAMENTO DIRETTO DEL FILE PARQUET E CALCOLO CARTELLA TARGET ---
parquet_path = '/.venv/Data/merged_adm1_wide_con_coordinate.parquet'
target_data_dir = os.path.dirname(parquet_path) # Questo estrae: '/Users/mattiadetommaso/PyCharmMiscProject/.venv/Data/'

print(f"Loading geolocation data from: {parquet_path}")
print(f"Target directory for output CSV: {target_data_dir}")
df_geo = pd.read_parquet(parquet_path)

# Prefisso per i grafici (rimane dentro la cartella dei risultati)
global_nat_prefix = os.path.join(results_dir, "global", "global_national")

# Identifica dinamicamente il nome della colonna del paese nel file Parquet
country_col_name = None
possible_country_cols = ["country", "country_code", "iso_a3", "iso_code", "country_id"]

for col in df_geo.columns:
    if col.lower() in possible_country_cols:
        country_col_name = col
        print(f"-> Found country column in Parquet: '{country_col_name}'")
        break

if country_col_name is None:
    print("-> Warning: Nessuna colonna 'country' esplicita trovata nel Parquet. Userò df_features come fallback per le coordinate.")

# Aggregate regional time series and coordinates to national level
national_series = {}
national_meta = []

for c_code, group in df_meta.groupby("country"):
    valid_dfs = []
    for _, row in group.iterrows():
        valid_dfs.append(regions_data[row["key"]])

    # Align to a common monthly range for this country's regions
    common_idx = valid_dfs[0].index
    for s in valid_dfs[1:]:
        common_idx = common_idx.intersection(s.index)

    if len(common_idx) >= 24:
        aligned_series = [s.reindex(common_idx) for s in valid_dfs]
        nat_series = pd.concat(aligned_series, axis=1).mean(axis=1)
        national_series[c_code] = nat_series

        # Estrazione delle coordinate
        mean_lat, mean_lon = None, None
        if country_col_name is not None:
            c_geo = df_geo[df_geo[country_col_name] == c_code]
            if not c_geo.empty and "latitude" in c_geo.columns and "longitude" in c_geo.columns:
                mean_lat = c_geo["latitude"].mean()
                mean_lon = c_geo["longitude"].mean()

        if mean_lat is None or pd.isna(mean_lat):
            c_feats = df_features[df_features["country"] == c_code]
            mean_lat = c_feats["latitude"].mean() if not c_feats.empty else 0.0
            mean_lon = c_feats["longitude"].mean() if not c_feats.empty else 0.0

        # Calculate features on the aggregated national series
        feats = extract_structural_features(nat_series)
        feats.update({
            "country": c_code,
            "adm1_pcode": c_code,
            "region_name": c_code,
            "latitude": mean_lat,
            "longitude": mean_lon
        })
        national_meta.append(feats)

df_nat_features = pd.DataFrame(national_meta)
print(f"Countries with aggregated national time series (>= 24 months): {len(df_nat_features)}")

if len(df_nat_features) >= 4:
    nat_results = run_clustering_flow(df_nat_features, national_series, global_nat_prefix, n_clusters=4, is_global=False, feature_cols=feature_cols)

    # --- MODIFICA SALVATAGGIO CSV ---
    # Il file viene salvato direttamente in '/Users/mattiadetommaso/PyCharmMiscProject/.venv/Data/global_national_labels.csv'
    output_csv_path = os.path.join(target_data_dir, "global_national_labels.csv")
    nat_results["labels_df"].to_csv(output_csv_path, index=False)
    print(f"-> SUCCESS: CSV salvato correttamente in: {output_csv_path}")

    # Save national map
    try:
        world_url = 'https://raw.githubusercontent.com/datasets/geo-boundaries-world-110m/master/countries.geojson'
        world = gpd.read_file(world_url)
        world = world[world["continent"] != "Antarctica"]

        # Merge world geometry with national cluster labels
        world_merged = world.merge(nat_results["labels_df"], left_on="iso_a3", right_on="country", how="left")

        fig, ax = plt.subplots(figsize=(18, 10))
        world_merged.plot(color="#f2f2f2", edgecolor="#d9d9d9", ax=ax)
        world_merged.dropna(subset=["kmeans_features_with_coords"]).plot(
            column="kmeans_features_with_coords", cmap="tab10", legend=True, categorical=True,
            ax=ax, legend_kwds={"bbox_to_anchor": (1.05, 1), "loc": "upper left"}
        )
        ax.set_title("Global National-Level Clusters (PCA + K-Means - WITH Coordinates)", fontsize=16)
        ax.set_xlim([-120, 150])
        ax.set_ylim([-40, 60])
        plt.tight_layout()
        plt.savefig(global_nat_prefix + "_map_with_coords.png", dpi=150)
        plt.close()
        print("Global national maps plotted successfully.")
    except Exception as e:
        print(f"Global national map plotting failed: {e}")

# Save comparative metrics summary
df_metrics = pd.DataFrame(metrics_records)
df_metrics.to_csv(os.path.join(results_dir, "country_level_clustering_silhouette_scores.csv"), index=False)
print("\nSaved comparative silhouette scores summary CSV.")
print("\n--- Pipeline Completed Successfully! ---")


--- Step 6: Running Global National-Level Clustering ---
Loading geolocation data from: /Users/mattiadetommaso/PyCharmMiscProject/.venv/Data/merged_adm1_wide_con_coordinate.parquet
Target directory for output CSV: /Users/mattiadetommaso/PyCharmMiscProject/.venv/Data
-> Found country column in Parquet: 'Country'
Countries with aggregated national time series (>= 24 months): 35
-> SUCCESS: CSV salvato correttamente in: /Users/mattiadetommaso/PyCharmMiscProject/.venv/Data/global_national_labels.csv
Global national map plotting failed: Invalid projection: EPSG:4326: (Internal Proj Error: proj_create: no database context specified)

Saved comparative silhouette scores summary CSV.

--- Pipeline Completed Successfully! ---
